# Corrected checkpoint-selection comparison

This notebook reevaluates every saved LoRA checkpoint with the publication EOT/parser contract. It loads each BF16 base model once, swaps adapter checkpoints in memory, never loads merged/FP8/NVFP4 exports, preserves the old reports, and reports whether each selected checkpoint changed. Training is not run.

In [ ]:
# Cell 1: mount Drive, stage the flat or nested package, and install BF16 evaluation dependencies
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_BUNDLE_DIR = Path('/content/drive/MyDrive/colab_drive_bundle')
NESTED_PACKAGE_DIR = DRIVE_BUNDLE_DIR / 'fen_move_colab'
PACKAGE_ON_DRIVE = (
    NESTED_PACKAGE_DIR
    if (NESTED_PACKAGE_DIR / '__init__.py').is_file()
    else DRIVE_BUNDLE_DIR
)
WORKSPACE = Path('/content/checkpoint_selection_workspace')

if not (PACKAGE_ON_DRIVE / '__init__.py').is_file():
    raise FileNotFoundError(f'No package __init__.py found below {DRIVE_BUNDLE_DIR}')
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir(parents=True)
shutil.copytree(
    PACKAGE_ON_DRIVE,
    WORKSPACE / 'fen_move_colab',
    ignore=shutil.ignore_patterns(
        'raw', 'datasets', 'runs', 'vllm_exports',
        'publication_evaluation_v1', 'parallelism_benchmarks_v1',
        '__pycache__', '.ipynb_checkpoints', '.git',
    ),
)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))

try:
    token = userdata.get('HF_TOKEN') or userdata.get('hf_token')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

install_env = os.environ.copy()
install_env['INSTALL_LOW_PRECISION'] = '0'
subprocess.check_call(
    ['bash', str(WORKSPACE / 'fen_move_colab' / 'install_colab.sh')],
    env=install_env,
)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
print(f'Package source: {PACKAGE_ON_DRIVE}')


In [ ]:
# Cell 2: the six completed training runs
RUN_NAMES = [
    'gemma3_270m_all_legal_moves_v3',  # rank 32
    'gemma3_270m_all_legal_moves_v4',  # rank 16
    'gemma4_e2b_all_legal_moves_v3',   # rank 32
    'gemma4_e2b_all_legal_moves_v4',   # rank 16
    'gemma4_e4b_all_legal_moves_v3',   # rank 32
    'gemma4_e4b_all_legal_moves_v4',   # rank 16
]
VALIDATION_SAMPLES = 128
VALIDATION_BATCH_SIZE = 8
VALIDATION_MAX_NEW_TOKENS = 512

for run_name in RUN_NAMES:
    run_path = DRIVE_BUNDLE_DIR / 'runs' / run_name
    if not run_path.is_dir():
        raise FileNotFoundError(f'Missing configured run: {run_path}')
print(f'Ready to compare {len(RUN_NAMES)} runs. No merged or quantized exports will be loaded.')


In [ ]:
# Cell 3: run the resumable, shared-base comparison
from fen_move_colab.compare_checkpoint_selections import compare_checkpoint_selections

SUMMARY = compare_checkpoint_selections(
    drive_bundle_dir=str(DRIVE_BUNDLE_DIR),
    run_names=RUN_NAMES,
    samples=VALIDATION_SAMPLES,
    batch_size=VALIDATION_BATCH_SIZE,
    max_new_tokens=VALIDATION_MAX_NEW_TOKENS,
)


In [ ]:
# Cell 4: compact decision table
import pandas as pd

display(pd.DataFrame(SUMMARY['results'])[[
    'run_name',
    'checkpoints_evaluated',
    'old_best_f1',
    'new_best_f1',
    'best_checkpoint_changed',
    'old_best_checkpoint',
    'new_best_checkpoint',
]])
print('Runs requiring re-export:', SUMMARY['changed_runs'])
